# Colab setup
Run these cells top to bottom at the start of EVERY session. Nothing here is permanent except what is on Drive and GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT THIS: your GitHub repo URL
REPO_URL = "https://github.com/arooj-zehra/xai-fire-deforestation.git"
DRIVE_DIR = "/content/drive/MyDrive/xai-fire-deforestation"   # data zip + outputs live here
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
%cd /content
import os
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo pull
else:
    !git clone {REPO_URL} /content/repo
%cd /content/repo
!pip install -q grad-cam shap

In [ ]:
# Unpack the preprocessed dataset zip from Drive to fast local disk (about 1 minute)
import os
if not os.path.isdir('/content/data/processed'):
    !mkdir -p /content/data
    !unzip -q "{DRIVE_DIR}/dataset_224.zip" -d /content/data/processed
!ls /content/data/processed | head

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
# Checkpoints/metrics: always write to DRIVE_DIR/outputs so they survive disconnects.

## Phase 2: baseline training
Results go to Drive so they survive disconnects. Re-run the cells above first in a fresh session.

In [ ]:
%cd /content/repo
!git pull
!python -m src.train --data /content/data/processed --out "{DRIVE_DIR}/outputs" --name baseline --epochs 12 --freeze-epochs 2

## Phase 3: imbalance + augmentation grid
A = `baseline` (already done). Runs below are B-F. Each run is skipped if its metrics.json already exists on Drive, so after a disconnect just re-run the setup cells and this cell. About 8-9 min per run on a T4.

In [ ]:
%cd /content/repo
!git pull
import os
grid = {
    "B_aug":            "--aug",
    "C_wloss":          "--imbalance weighted_loss",
    "D_wloss_aug":      "--imbalance weighted_loss --aug",
    "E_sampler":        "--imbalance sampler",
    "F_sampler_aug":    "--imbalance sampler --aug",
}
for name, flags in grid.items():
    if os.path.exists(f"{DRIVE_DIR}/outputs/{name}/metrics.json"):
        print("skip (done):", name); continue
    print("=== running", name, flags)
    !python -m src.train --data /content/data/processed --out "{DRIVE_DIR}/outputs" --name {name} --epochs 12 --freeze-epochs 2 {flags}

In [ ]:
!python scripts/summarize_runs.py --outputs "{DRIVE_DIR}/outputs"

## Phase 3b: second architecture (REQUIRED) with the winning config (B_aug)

In [ ]:
%cd /content/repo
!git pull
import os
name = "G_resnet18_aug"
if os.path.exists(f"{DRIVE_DIR}/outputs/{name}/metrics.json"):
    print("skip (done):", name)
else:
    !python -m src.train --data /content/data/processed --out "{DRIVE_DIR}/outputs" --name {name} --model resnet18 --aug --epochs 12 --freeze-epochs 2
!python scripts/summarize_runs.py --outputs "{DRIVE_DIR}/outputs"

## Phase 3c (OPTIONAL, ~16 min): repeat baseline and B_aug with another seed to check the gain is not noise

In [ ]:
%cd /content/repo
import os
for name, flags in {"baseline_s7": "", "B_aug_s7": "--aug"}.items():
    if os.path.exists(f"{DRIVE_DIR}/outputs/{name}/metrics.json"):
        print("skip (done):", name); continue
    !python -m src.train --data /content/data/processed --out "{DRIVE_DIR}/outputs" --name {name} --seed 7 --epochs 12 --freeze-epochs 2 {flags}
!python scripts/summarize_runs.py --outputs "{DRIVE_DIR}/outputs"

## Phase 4: shortcut / per-source check on the final model (B_aug)
Compares the CNN with a source-only baseline and a colour-statistics baseline. Takes about 3-5 minutes.

In [ ]:
%cd /content/repo
!git pull
!python scripts/source_report.py --data /content/data/processed --run "{DRIVE_DIR}/outputs/B_aug"

## Phase 5: explainability (Grad-CAM + SHAP) for the final model (B_aug)
Writes figures to `outputs/B_aug/explain/` on Drive. A few minutes on a T4.

In [ ]:
%cd /content/repo
!git pull
!python -m src.run_explain --data /content/data/processed --run "{DRIVE_DIR}/outputs/B_aug"

In [ ]:
from IPython.display import Image, display
display(Image(f"{DRIVE_DIR}/outputs/B_aug/explain/explanations.png"))
display(Image(f"{DRIVE_DIR}/outputs/B_aug/explain/shap_channels.png"))